# Test multisigma pour la marée 

In [ ]:
# Set up a local cluster for distributed computing.
from distributed import LocalCluster

cluster = LocalCluster()
client = cluster.get_client()
client

In [ ]:
import fsspec
import s3fs
import numpy as np
import xdggs
import xarray as xr
import pandas as pd

In [ ]:
storage_options = {
    "anon": False,
    "profile": "gfts",
    "client_kwargs": {
        "endpoint_url": "https://s3.gra.perf.cloud.ovh.net",
        "region_name": "gra",
    },
}

fs = s3fs.S3FileSystem(
    anon=storage_options["anon"],
    profile=storage_options["profile"],
    client_kwargs=storage_options["client_kwargs"],
)

tag_name = "A15789"
scratch_root = "s3://gfts-ifremer/eel/run/chue/test"
target_root = f"{scratch_root}/{tag_name}"

default_chunk = {"time": 24}

In [ ]:
combined_diff_bathy = xr.open_dataset(
    f"{target_root}/emission_w_tide_{tag_name}.zarr", engine="zarr"
)
combined_diff_bathy.compute()

In [ ]:
emission_tide = xr.open_dataset(
    "/home/jovyan/pangeo-fish-old/notebooks/Tide/A15789_tide_pdf_healpix.zarr",
    engine="zarr",
)
emission = xr.open_dataset(
    f"{target_root}/emission_w_bathy_pdf_{tag_name}.zarr", engine="zarr"
)

In [ ]:
bathy_pdf = xr.open_dataset(
    f"{target_root}/bathy_pdf.zarr",
    engine="zarr",
    chunks={"time": 100, "cells": 10000},
    inline_array=True,
    storage_options=storage_options,
)

bathy_pdf

In [ ]:
emission_tide

In [ ]:
combined_diff_bathy["tide_found"].isel(time=slice(750, 770)).values

In [ ]:
states

In [ ]:
states.states.compute().dggs.decode(
    {"grid_name": "healpix", "level": 12, "indexing_scheme": "nested"}
).dggs.explore(alpha=0.8)

In [ ]:
import ipywidgets as ipw

In [ ]:
m1 = (
    emission.pdf.compute()
    .dggs.decode({"grid_name": "healpix", "level": 12, "indexing_scheme": "nested"})
    .dggs.explore(alpha=0.8)
)
m2 = (
    emission_tide.pdf.compute()
    .dggs.decode({"grid_name": "healpix", "level": 12, "indexing_scheme": "nested"})
    .dggs.explore(alpha=0.8)
)
m3 = (
    bathy_pdf.pdf_bathy.compute()
    .dggs.decode({"grid_name": "healpix", "level": 12, "indexing_scheme": "nested"})
    .dggs.explore(alpha=0.8)
)

In [ ]:
def extract_slider(map_):
    return map_.children[1].children[0].children[0]

In [ ]:
ipw.jslink((extract_slider(m1), "value"), (extract_slider(m2), "value"))
ipw.jslink((extract_slider(m1), "value"), (extract_slider(m3), "value"))

m1 | m2 | m3

In [ ]:
import json  # ← nécessaire

with fsspec.open(
    f"/home/jovyan/pangeo-fish-old/notebooks/nouveau_tag/{tag_name}/parameters.json",
    "r",
    **{} if not target_root.startswith("s3://") else storage_options,
) as file:
    params = json.load(file)  # ← CORRECT
params

## Predict positions original

In [ ]:
def predict_positions(
    *,
    emission,  # "dataset avec tide_found"
    params,  # "json_parameters avec sigmas (liste de sigmas)"
    target_root: str,
    storage_options: dict,
    chunks: dict,
    track_modes=["mean", "mode"],
    additional_track_quantities=["speed", "distance"],
    save=True,
    **kwargs,
):
    """High-level helper function for predicting fish's positions and generating the consequent trajectories.
    It futhermore saves the latter under ``states.zarr`` and ``trajectories.parq``.

    .. warning::
        ``target_root`` must not end with "/".

    .. note::
        If the estimator to load has multiple sigma values, their indices are expected to be defined in the entry "predictor_index" (``emission["predictor_index"]``).

    Parameters
    ----------
    target_root : str
        Path to a folder that must contain a folder ``combined.zarr`` and the file ``parameters.json``
    storage_options : dict
        Additional information for ``xarray`` to open the ``.zarr`` array
    chunks : dict
        Chunk size to load the xarray.Dataset ``combined.zarr``
    track_modes : list of str, default: ["mean", "mode"]
        Options for decoding trajectories.
    additional_track_quantities : list of str, default: ["speed", "distance"]
        Additional quantities to compute from the decoded tracks.
    save : bool, default: True
        Whether to save the ``states`` distribution and the trajectories.

    Returns
    -------
    states : xarray.Dataset
        The positional temporal probabilities. In case of multi-sigma optimization, it also adds the variable "sigma".
    trajectories : movingpandas.TrajectoryCollection
        The tracks decoded from `states`

    See Also
    --------
    pangeo_fish.hmm.estimator.EagerEstimator.decode
    """

    predictor_index = "predictor_index"

    emission = emission.compute()

    if "cells" in emission.dims:
        emission = to_healpix(emission)

    ## deals with the different possible data sources of the "predictor_index" integer values
    # one parameter
    if len(params["sigmas"]) == 1:
        emission = emission.assign(
            predictor_index=("time", np.zeros(emission["time"].size).astype(np.int32))
        )
        warnings.warn(
            'A "predictor_index" entry (filled with 0) is added to the loaded `emission` dataset.',
            RuntimeWarning,
        )
    # more than one parameter
    else:
        # in the input dataset
        if tide_found in emission:
            index_data_type = emission[tide_found].dtype
            if index_data_type != np.int32:
                emission[tide_found] = emission[tide_found].astype(np.int32)
                warnings.warn(
                    f'Entry "predictor_index" in the loaded `emission` dataset is cast to `np.int32` (found "{index_data_type}").',
                    RuntimeWarning,
                )
        # last attempt before raising an error: restore the sigma variable from the dictionary of parameters
        elif "sigma_indices" in params:

            def _compute_predictor_indices(indices: list[list[int]]):
                # TODO: input checking...
                size = sum([len(sub_list) for sub_list in indices])
                acc = [None for _ in range(size)]

                for i in range(len(indices)):
                    for index in indices[i]:
                        acc[index] = i

                assert all([s is not None for s in acc])
                return np.array(acc).astype(np.int32)

            try:
                emission = emission.assign(
                    predictor_index=(
                        "time",
                        _compute_predictor_indices(params["sigma_indices"]),
                    )
                )
                warnings.warn(
                    'Entry "predictor_index" not found in the emission dataset: predictors\' indices were restored from the dictionary `params`.',
                    RuntimeWarning,
                )
            except Exception:
                raise ValueError(
                    'Entry "predictor_index" is missing in the emission dataset and predictors\' indices could not be restored from the dictionary `params`.'
                )
        else:
            raise ValueError(
                'The time indices for each sigma value is not defined in the `emission` (entry "predictor_index" is missing) and no indices found in the dictionary `params` (entry "sigma_indices" is missing).'
            )

    # do not account for the other kwargs...
    # not very robust yet...
    truncate = float(params["predictor_factory"]["kwargs"]["truncate"])
    cls_name = params["predictor_factory"]["class"]  # type: str
    if "Gaussian2DCartesian" in cls_name:
        predictor_factory = _get_predictor_factory(
            emission, truncate=truncate, dims=["x", "y"]
        )
    elif "Gaussian1DHealpix" in cls_name:
        predictor_factory = _get_predictor_factory(
            emission, truncate=truncate, dims=["cells"]
        )
    else:
        raise RuntimeError("Could not infer predictor's class from the `.json` file.")

    optimized = EagerEstimator(
        sigmas=params["sigmas"], predictor_factory=predictor_factory
    )

    states = optimized.predict_proba(emission)  # type: xr.DataArray
    states = (
        states.to_dataset()
        .chunk(chunks)
        .assign_attrs(
            emission.attrs | _get_package_versions() | {"sigmas": params["sigmas"]}
        )
    )  # type: xr.Dataset

    # adds the variable `sigma` to `states`
    if len(params["sigmas"]) > 1:
        # from the indices in emission
        if predictor_index in emission:
            sigma_indices = [
                list(group_indices)
                for group_indices in emission.groupby(predictor_index).groups.values()
            ]
        elif "sigma_indices" in params:
            sigma_indices = _compute_predictor_indices(params["sigma_indices"])
        else:
            raise RuntimeError(
                "Failed to add the variable `sigma` to `states` (it should never happen.)"
            )

        sigma_var = _compute_sigma_var(sigma_indices, params["sigmas"])
        states = states.assign(sigma=("time", sigma_var))

    if save:
        _save_zarr(states, f"{target_root}/states.zarr", storage_options)

    trajectories = optimized.decode(
        emission,
        states.fillna(0),
        mode=track_modes,
        progress=False,
        additional_quantities=additional_track_quantities,
    )

    if save:
        save_trajectories(trajectories, target_root, storage_options, format="parquet")

    return states, trajectories

## Predict position avec tide

In [ ]:
import numpy as np
import warnings
import xarray as xr
from pangeo_fish.hmm.estimator import EagerEstimator
from toolz.functoolz import curry
from pangeo_fish.hmm.prediction import Gaussian1DHealpix, Gaussian2DCartesian
from xdggs import HealpixInfo
from pangeo_fish.helpers import _get_package_versions


def _get_predictor_factory(ds: xr.Dataset, truncate: float, dims: list[str]):
    if dims == ["x", "y"]:
        predictor = curry(Gaussian2DCartesian, truncate=truncate)
    elif dims == ["cells"]:
        predictor = curry(
            Gaussian1DHealpix,
            cell_ids=ds["cell_ids"].data,
            grid_info=ds.dggs.grid_info,
            truncate=truncate,
            weights_threshold=1e-8,
            pad_kwargs={"mode": "constant", "constant_value": 0},
            optimize_convolution=True,
        )
    else:
        raise ValueError(f'Unknown dims "{dims}".')
    return predictor

In [ ]:
def predict_positions_tide(
    *,
    emission,  # dataset avec pdf + tide_found
    params,  # json_parameters avec sigmas
    target_root: str,
    storage_options: dict,
    chunks: dict,
    track_modes=["mean", "mode"],
    additional_track_quantities=["speed", "distance"],
    save=True,
    **kwargs,
):

    # calculer si nécessaire
    emission = emission.compute().drop_indexes("cells")

    # convertir en healpix si nécessaire
    if "cells" not in emission.dims:
        emission = to_healpix(emission)

    # créer predictor_index selon tide_found
    if "tide_found" not in emission:
        raise ValueError('Le dataset doit contenir la variable "tide_found"')

    # sigmas
    sigmas = params["0"]["sigmas"]

    # créer predictor_index : 0 pour tide_found=0, 1 pour tide_found=1
    predictor_index = emission["tide_found"].astype(np.int32)
    if not set(np.unique(predictor_index.values)).issubset({0, 1}):
        raise ValueError("tide_found doit contenir uniquement 0 ou 1")

    emission = emission.assign(
        predictor_index=("time", predictor_index.values.astype(np.int32))
    )
    grid_info = HealpixInfo(level=12, indexing_scheme="nested")
    # créer le predictor_factory selon la classe
    truncate = float(4)  # float(params["predictor_factory"]["kwargs"]["truncate"])
    cls_name = "Gaussian1DHealpix"  # params["predictor_factory"]["class"]  # type: str
    print("choix de la methode", cls_name)
    if "Gaussian2DCartesian" in cls_name:
        predictor_factory = _get_predictor_factory(
            emission, truncate=truncate, dims=["x", "y"]
        )
    elif "Gaussian1DHealpix" in cls_name:
        predictor_factory = _get_predictor_factory(
            emission, truncate=truncate, dims=["cells"]
        )
    else:
        raise RuntimeError("Impossible d'inférer la classe du predictor.")

    # créer l'estimateur avec les deux sigmas
    optimized = EagerEstimator(sigmas=sigmas, predictor_factory=predictor_factory)

    # prédire la distribution d'état
    states = optimized.predict_proba(emission)  # xr.DataArray

    # return states

    states = states.to_dataset().chunk(chunks)
    states = states.assign_attrs(
        emission.attrs | _get_package_versions() | {"sigmas": sigmas}
    )

    # ajouter sigma correspondant à chaque time
    sigma_var = np.array([sigmas[i] for i in predictor_index.values])
    states = states.assign(sigma=("time", sigma_var))

    # sauvegarde
    if save:
        _save_zarr(states, f"{target_root}/states.zarr", storage_options)

    # décoder les trajectoires
    # trajectories = optimized.decode(
    #    emission,
    #    states.fillna(0),
    #    mode=track_modes,
    #    progress=False,
    #    additional_quantities=additional_track_quantities,
    # )

    # if save:
    #    save_trajectories(trajectories, target_root, storage_options, format="parquet")

    return states  # , trajectories

In [ ]:
def predict_positions_tide(
    *,
    emission,  # dataset avec pdf + tide_found
    params,  # json_parameters avec sigmas
    target_root: str,
    storage_options: dict,
    chunks: dict,
    track_modes=["mean", "mode"],
    additional_track_quantities=["speed", "distance"],
    save=True,
    **kwargs,
):

    # calculer si nécessaire
    emission = emission.compute().drop_indexes("cells")

    # convertir en healpix si nécessaire
    if "cells" not in emission.dims:
        emission = to_healpix(emission)

    # créer predictor_index selon tide_found
    if "tide_found" not in emission:
        raise ValueError('Le dataset doit contenir la variable "tide_found"')

    # sigmas
    sigmas = params["0"]["sigmas"]

    # créer predictor_index : 0 pour tide_found=0, 1 pour tide_found=1
    predictor_index = emission["tide_found"].astype(np.int32)
    if not set(np.unique(predictor_index.values)).issubset({0, 1}):
        raise ValueError("tide_found doit contenir uniquement 0 ou 1")

    emission = emission.assign(
        predictor_index=("time", predictor_index.values.astype(np.int32))
    )
    grid_info = HealpixInfo(level=12, indexing_scheme="nested")
    # créer le predictor_factory selon la classe
    truncate = float(4)  # float(params["predictor_factory"]["kwargs"]["truncate"])
    cls_name = "Gaussian1DHealpix"  # params["predictor_factory"]["class"]  # type: str
    print("choix de la methode", cls_name)
    if "Gaussian2DCartesian" in cls_name:
        predictor_factory = _get_predictor_factory(
            emission, truncate=truncate, dims=["x", "y"]
        )
    elif "Gaussian1DHealpix" in cls_name:
        predictor_factory = _get_predictor_factory(
            emission, truncate=truncate, dims=["cells"]
        )

    else:
        raise RuntimeError("Impossible d'inférer la classe du predictor.")

    # créer l'estimateur avec les deux sigmas
    optimized = EagerEstimator(sigmas=sigmas, predictor_factory=predictor_factory)
    print(optimized)
    # prédire la distribution d'état
    states = optimized.predict_proba(emission)  # xr.DataArray

    return states

In [ ]:
combined_diff_bathy = combined_diff_bathy.dggs.decode(
    {"grid_name": "healpix", "level": 12, "indexing_scheme": "nested"}
)

states = predict_positions_tide(
    emission=combined_diff_bathy,
    params=params,
    target_root=f"{target_root}",
    storage_options=storage_options,
    chunks=default_chunk,
    save=False,
)

In [ ]:
states = states.to_dataset().chunk(chunks)
states = states.assign_attrs(
    emission.attrs | _get_package_versions() | {"sigmas": sigmas}
)

# ajouter sigma correspondant à chaque time
sigma_var = np.array([sigmas[i] for i in predictor_index.values])
states = states.assign(sigma=("time", sigma_var))

# sauvegarde
if save:
    _save_zarr(states, f"{target_root}/states.zarr", storage_options)

# décoder les trajectoires
# trajectories = optimized.decode(
#    emission,
#    states.fillna(0),
#    mode=track_modes,
#    progress=False,
#    additional_quantities=additional_track_quantities,
# )

# if save:
#    save_trajectories(trajectories, target_root, storage_options, format="parquet")

return states  # , trajectories

In [ ]:
states.states.dggs.decode(
    {"grid_name": "healpix", "level": 12, "indexing_scheme": "nested"}
).dggs.explore(alpha=0.8)

In [ ]:
states.to_zarr(
    "/home/jovyan/pangeo-fish/notebooks/nouveau_tag/A15789/states_sigmas_60_200.zarr",
    mode="w",
    consolidated=True,
)

In [ ]:
import xarray as xr

states = xr.open_dataset(
    "/home/jovyan/pangeo-fish/notebooks/nouveau_tag/A15789/states_sigmas_30_100.zarr"
)
states

In [ ]:
combined_diff_bathy = combined_diff_bathy.dggs.decode(
    {"grid_name": "healpix", "level": 12, "indexing_scheme": "nested"}
)

states = predict_positions_tide(
    emission=combined_diff_bathy,
    params=params,
    target_root=f"{target_root}",
    storage_options=storage_options,
    chunks=default_chunk,
    save=False,
)

In [ ]:
states.compute().to_zarr("states_tide.zarr", mode="w")

In [ ]:
states